In [1]:
import pandas as pd  

dvf = pd.read_csv("../data/raw/dvf_69.csv", sep= ',', dtype={ "numero_volume": "string", "lot1_numero": "string"})

print('Dimension de la base :', dvf.shape)
print("\nInformations :"); dvf.info()
print("\nDescription des variables :"); display(dvf.describe())
print("\nListe des colonnes :"); print(dvf.columns.tolist())
print("\nTypes de variables :"); display(dvf.dtypes.value_counts())
print("\nDoublons :"); print(dvf.duplicated().sum())
display(dvf.head())

# Colonnes NA; type variables; taux
documentation_variables = pd.DataFrame({
    "Valeurs manquantes": dvf.isnull().sum(), 
    "Taux de NA (%)": (dvf.isna().mean() * 100).round(2).values,
    "Type informatique": dvf.dtypes.values
})
print("\nDocumentation des variables :"); display(documentation_variables)

# Colonne distribution des valeurs
distribution = {}
liste = {}
for col in dvf.columns:  
    if dvf[col].dtype in ["object", "string", "str"]:
        
        if dvf[col].nunique() <= 10:
            distribution[col] = ", ".join(dvf[col].dropna().astype(str).unique())
            liste[col]= "liste des catégories"
        else:
            distribution[col] = f"{dvf[col].nunique()} catégories"
            liste[col]= "nbr catégories"
    
    else:
        # distribution.append("Voir statistiques descriptives")
        describe = dvf[col].describe()
        liste[col]= "Voir statistiques descriptives"
        distribution[col] = (
            f"Min={describe['min']:.2f} | "
            f"Médiane={describe['50%']:.2f} | "
            f"Moyenne={describe['mean']:.2f} | "
            f"Max={describe['max']:.2f}"
        )

documentation_distribution = pd.DataFrame({
    "Nom de la colonne": dvf.columns,
    "Type": dvf.dtypes.values,
    "Liste": [liste[col] for col in dvf.columns],
    "Distribution des valeurs": [distribution[col] for col in dvf.columns]
})

pd.set_option("display.max_colwidth", None) # affiche le contenu complet des cellules
print("\nDocumentation distribution des valeurs :"); display(documentation_distribution)

# Autres analyses:
print("\nNbr des pièces: ")
display(dvf["nombre_pieces_principales"].value_counts().sort_index())

print("Mutations dupliquées :") ;display(dvf["id_mutation"].duplicated().sum())

print("\nLa répartition des types de biens: ")
display(dvf["type_local"].value_counts(normalize=True)*100)

print("\nCorrélation des variables numériques avec 'valeur_fonciere' :")
display(dvf.corr(numeric_only=True)["valeur_fonciere"].sort_values(ascending=False))

print("\nTransactions avec une valeur foncière inférieure à 1 000 € :")
display(dvf[dvf["valeur_fonciere"] < 1_000])

print("\nTransactions avec une valeur foncière supérieure à 10 000 000 € :")
display(dvf[dvf["valeur_fonciere"] > 10_000_000])

# Vérifier les valeurs aberrantes
print("\nMaisons ou appartements avec 0 pièce :")
display(dvf[(dvf["type_local"].isin(["Maison", "Appartement"])) & (dvf["nombre_pieces_principales"] == 0)])

print("Surfaces nulles ou négatives :") ;display(dvf[dvf["surface_reelle_bati"] <= 0])
print("Terrains nuls :") ; display(dvf[dvf["surface_terrain"] <= 0])

#nbr de mutation par ans
dvf["date_mutation"] = pd.to_datetime(dvf["date_mutation"])
print("Nombre de mutations par année :") ; display(dvf["date_mutation"].dt.year.value_counts().sort_index())

#comparer les descriptions des types de biens 
print("Valeur foncière selon le type de local :")
display(dvf.groupby("type_local")["valeur_fonciere"].agg(["count","mean","median","min","max"]).sort_values("count", ascending=False))

# Prix au m2
filtre = dvf["surface_reelle_bati"] > 0
prix_m2 = dvf.loc[filtre, "valeur_fonciere"] / dvf.loc[filtre, "surface_reelle_bati"]
print("Description du prix au m² :"); display(prix_m2.describe())



Dimension de la base : (462796, 40)

Informations :
<class 'pandas.DataFrame'>
RangeIndex: 462796 entries, 0 to 462795
Data columns (total 40 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id_mutation                   462796 non-null  str    
 1   date_mutation                 462796 non-null  str    
 2   numero_disposition            462796 non-null  int64  
 3   nature_mutation               462796 non-null  str    
 4   valeur_fonciere               460759 non-null  float64
 5   adresse_numero                374320 non-null  float64
 6   adresse_suffixe               24966 non-null   str    
 7   adresse_nom_voie              456327 non-null  str    
 8   adresse_code_voie             456333 non-null  str    
 9   code_postal                   456333 non-null  float64
 10  code_commune                  462796 non-null  int64  
 11  nom_commune                   462796 non-null  str    
 12  cod

,numero_disposition,valeur_fonciere,adresse_numero,code_postal,code_commune,code_departement,ancien_code_commune,ancien_nom_commune,ancien_id_parcelle,lot1_surface_carrez,...,lot4_surface_carrez,lot5_numero,lot5_surface_carrez,nombre_lots,code_type_local,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude
count,462796.000000,4.607590e+05,374320.000000,456333.000000,462796.000000,462796.0,0.0,0.0,0.0,88161.000000,...,627.000000,1539.000000,196.000000,462796.000000,367240.000000,188184.000000,366921.000000,1.698760e+05,447339.000000,447339.000000
mean,1.095839,7.341485e+05,310.959308,69292.350301,69231.847142,69.0,NaN,NaN,NaN,72.089200,...,108.648692,61.719298,98.505867,0.860427,2.542136,108.442089,1.399342,1.785775e+03,4.795316,45.790087
std,1.136890,2.523503e+06,1249.810137,263.921648,122.286363,0.0,NaN,NaN,NaN,119.264941,...,198.670618,320.775295,73.564085,0.891742,0.806044,540.156226,1.882392,2.252921e+04,0.134899,0.124133
min,1.000000,1.800000e-01,1.000000,69001.000000,69001.000000,69.0,NaN,NaN,NaN,0.570000,...,5.100000,2.000000,3.310000,0.000000,1.000000,1.000000,0.000000,1.000000e+00,4.249090,45.457109
25%,1.000000,1.550000e+05,11.000000,69009.000000,69122.000000,69.0,NaN,NaN,NaN,42.660000,...,55.120000,7.000000,39.000000,0.000000,2.000000,45.000000,0.000000,2.320000e+02,4.742462,45.734446
50%,1.000000,2.671220e+05,31.000000,69220.000000,69260.000000,69.0,NaN,NaN,NaN,63.070000,...,87.590000,23.000000,88.235000,1.000000,3.000000,68.000000,0.000000,5.490000e+02,4.832204,45.763065
75%,1.000000,4.600000e+05,90.000000,69480.000000,69381.000000,69.0,NaN,NaN,NaN,80.790000,...,118.900000,54.000000,139.510000,1.000000,3.000000,95.000000,3.000000,1.252000e+03,4.873844,45.805527
max,82.000000,2.361600e+08,9999.000000,69970.000000,69389.000000,69.0,NaN,NaN,NaN,7570.000000,...,2455.000000,8371.000000,442.600000,83.000000,4.000000,61640.000000,34.000000,7.700000e+06,5.159332,46.303584



Liste des colonnes :
['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation', 'valeur_fonciere', 'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune', 'nom_commune', 'code_departement', 'ancien_code_commune', 'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle', 'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez', 'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale', 'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude']

Types de variables :


float64    21
str        13
int64       4
string      2
Name: count, dtype: int64


Doublons :
39362


...

date_mutation
2021    108695
2022    108228
2023     84968
2024     75985
2025     84920
Name: count, dtype: int64

Valeur foncière selon le type de local :


,count,mean,median,min,max
type_local,,,,,
Dépendance,178043,5.299376e+05,260000.00,1.00,18217110.0
Appartement,115247,5.958592e+05,241000.00,0.50,28629286.0
Maison,42091,4.998486e+05,393762.13,1.00,17000000.0
Local industriel. commercial ou assimilé,31375,2.383897e+06,550000.00,0.18,173035968.0


Description du prix au m² :


count    1.878540e+05
mean     2.886107e+04
std      1.664217e+05
min      5.788712e-05
25%      2.954545e+03
50%      4.250000e+03
75%      6.220723e+03
max      3.431138e+06
dtype: float64

In [2]:
import pandas as pd

# Chargement des données INSEE du Rhône
df_insee_69 = pd.read_csv("../data/raw/insee_comparateur_territoires_69.csv")

print("Dimensions :", df_insee_69.shape)
display(df_insee_69.head())
df_insee_69.info()

# Exploration de la structure des données
display("##################### Exploration de la structure des données")
print("Nombre de communes :", df_insee_69["GEO"].nunique())

print("\nAnnées disponibles :")
print(sorted(df_insee_69["TIME_PERIOD"].unique()))

print("\nSources disponibles :")
print(df_insee_69["DS"].value_counts())

print("\nNombre d'indicateurs :", df_insee_69["TAB_MEASURE"].nunique())

print("\nIndicateurs disponibles :")
print(df_insee_69["TAB_MEASURE"].value_counts())

# Exploration des indicateurs disponibles
display("##################### Exploration des indicateurs disponibles")

indicateurs = (df_insee_69[["DS", "TAB_MEASURE", "TIME_PERIOD", "UNIT_MEASURE"]].drop_duplicates()
    .sort_values(["DS", "TAB_MEASURE", "TIME_PERIOD"])
)

display("Nombre d'indicateurs différents :", df_insee_69["TAB_MEASURE"].nunique())
display(indicateurs)

display("##################### Résumé des indicateurs disponibles")

resume_indicateurs = (
    df_insee_69
    .groupby(["DS", "TAB_MEASURE"])
    .agg(
        annee_min=("TIME_PERIOD", "min"),
        annee_max=("TIME_PERIOD", "max"),
        nb_communes=("GEO", "nunique"),
        nb_valeurs=("OBS_VALUE", "count")
    )
    .reset_index()
    .sort_values(["DS", "TAB_MEASURE"])
)

display(resume_indicateurs)


# Indicateurs retenus pour l'exploration
display("##################### Indicateurs retenus pour l'exploration")
variables_insee = [
    "MED_SL",
    "PR_MD60",
    "POP",
    "SUP",
    "DWELLINGS",
    "DWELLINGS_OCS_DW_MAIN",
    "DWELLINGS_OCS_DW_MAIN_TSH_100",
    "DWELLINGS_OCS_DW_SEC_DW_OCC",
    "DWELLINGS_OCS_DW_VAC",
    "UNIT_LOC"
]

# Sélection des indicateurs
df_selection = df_insee_69[df_insee_69["TAB_MEASURE"].isin(variables_insee)].copy()

# Nombre et pourcentage de valeurs manquantes par indicateur
manquants = (
    df_selection
    .groupby("TAB_MEASURE")["OBS_VALUE"]
    .agg(
        nb_lignes="size",
        nb_valeurs="count"
    )
)

manquants["nb_manquants"] = (manquants["nb_lignes"] - manquants["nb_valeurs"])
manquants["pct_manquants"] = (manquants["nb_manquants"] / manquants["nb_lignes"] * 100 ).round(2)
display(manquants)

# Vérification des années disponibles pour les indicateurs sélectionnés
display("##################### Vérification des années disponibles pour les indicateurs sélectionnés")
annees_variables = (
    df_selection
    .groupby("TAB_MEASURE")["TIME_PERIOD"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index(name="annees_disponibles")
)

display(annees_variables)

Dimensions : (19250, 11)


,GEO_REF,GEO_OBJECT,GEO,TIME_PERIOD,DS,TAB_MEASURE,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,UNIT_MULT,CONF_STATUS
0,2026,ARM,69381,2025,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,242.0,A,NaN,NaN,NaN
1,2026,ARM,69381,2025,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,169.0,A,NaN,NaN,NaN
2,2026,ARM,69381,2024,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,255.0,A,NaN,NaN,NaN
3,2026,ARM,69381,2024,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,184.0,A,NaN,NaN,NaN
4,2026,ARM,69381,2024,DS_FLORES_A5,UNIT_LOC_NUMBER_EMPL_E100T199,8.0,A,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 19250 entries, 0 to 19249
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   GEO_REF       19250 non-null  int64  
 1   GEO_OBJECT    19250 non-null  str    
 2   GEO           19250 non-null  int64  
 3   TIME_PERIOD   19250 non-null  int64  
 4   DS            19250 non-null  str    
 5   TAB_MEASURE   19250 non-null  str    
 6   OBS_VALUE     19105 non-null  float64
 7   OBS_STATUS    19250 non-null  str    
 8   UNIT_MEASURE  3300 non-null   str    
 9   UNIT_MULT     550 non-null    float64
 10  CONF_STATUS   550 non-null    str    
dtypes: float64(2), int64(3), str(6)
memory usage: 2.3 MB


'##################### Exploration de la structure des données'

Nombre de communes : 275

Années disponibles :
[np.int64(1968), np.int64(1975), np.int64(1982), np.int64(1990), np.int64(1999), np.int64(2007), np.int64(2012), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Sources disponibles :
DS
DS_RP_LOGEMENT_PRINC            4125
DS_FLORES_A5                    3850
DS_ETAT_CIVIL_NAIS_COMMUNES     2750
DS_ETAT_CIVIL_DECES_COMMUNES    2750
DS_RP_SERIE_HISTORIQUE          2750
DS_RP_MENAGES_COMP               825
DS_RP_EMPLOI_LR_PRINC            825
DS_RP_EMPLOI_LT_PRINC            825
DS_FILOSOFI_CC                   550
Name: count, dtype: int64

Nombre d'indicateurs : 28

Indicateurs disponibles :
TAB_MEASURE
LVB                              2750
DTH                              2750
POP                              2475
DWELLINGS_OCS_DW_MAIN_TSH_100     825
DWELLINGS                         825
DWELLINGS_OCS_DW_MAIN             825
D

...

,DS,TAB_MEASURE,annee_min,annee_max,nb_communes,nb_valeurs
0,DS_ETAT_CIVIL_DECES_COMMUNES,DTH,2016,2025,275,2750
1,DS_ETAT_CIVIL_NAIS_COMMUNES,LVB,2016,2025,275,2750
2,DS_FILOSOFI_CC,MED_SL,2023,2023,275,274
3,DS_FILOSOFI_CC,PR_MD60,2023,2023,275,131
4,DS_FLORES_A5,UNIT_LOC,2024,2024,275,275
5,DS_FLORES_A5,UNIT_LOC_ACTIVITY_AZ,2024,2024,275,275
6,DS_FLORES_A5,UNIT_LOC_ACTIVITY_BE,2024,2024,275,275
7,DS_FLORES_A5,UNIT_LOC_ACTIVITY_FZ,2024,2024,275,275
8,DS_FLORES_A5,UNIT_LOC_ACTIVITY_GU,2024,2024,275,275
9,DS_FLORES_A5,UNIT_LOC_ACTIVITY_OQ,2024,2024,275,275


"##################### Indicateurs retenus pour l'exploration"

,nb_lignes,nb_valeurs,nb_manquants,pct_manquants
TAB_MEASURE,,,,
DWELLINGS,825,825,0,0.00
DWELLINGS_OCS_DW_MAIN,825,825,0,0.00
DWELLINGS_OCS_DW_MAIN_TSH_100,825,825,0,0.00
DWELLINGS_OCS_DW_SEC_DW_OCC,825,825,0,0.00
DWELLINGS_OCS_DW_VAC,825,825,0,0.00
MED_SL,275,274,1,0.36
POP,2475,2475,0,0.00
PR_MD60,275,131,144,52.36
SUP,275,275,0,0.00


'##################### Vérification des années disponibles pour les indicateurs sélectionnés'

,TAB_MEASURE,annees_disponibles
0,DWELLINGS,"[2012, 2017, 2023]"
1,DWELLINGS_OCS_DW_MAIN,"[2012, 2017, 2023]"
2,DWELLINGS_OCS_DW_MAIN_TSH_100,"[2012, 2017, 2023]"
3,DWELLINGS_OCS_DW_SEC_DW_OCC,"[2012, 2017, 2023]"
4,DWELLINGS_OCS_DW_VAC,"[2012, 2017, 2023]"
5,MED_SL,[2023]
6,POP,"[1968, 1975, 1982, 1990, 1999, 2007, 2012, 2017, 2023]"
7,PR_MD60,[2023]
8,SUP,[2023]
9,UNIT_LOC,[2024]
